In [16]:
# Cleaner function for TfidfVectorizer

import re
import html

def clean_text(text):
    
    text = re.sub(r"\\n|\\t|\n|\t|\\", " ", text)
    # Deletes the escape characters
    
    text = re.sub(r"\s+", " ", text).strip()
    # Deletes the extra spaces
    
    Apos_dict={"n't":" not","'m":" am","'ll":" will","'d":" would","'ve":" have","'re":" are"}
    # didn't add "'s":is --> because "'s" is not "is" every time
    
    for key,value in Apos_dict.items():
        if key in text:
            text=text.replace(key,value)
    # Fix the words with apostrophe
    # Example: we'll --> we will
    
    return text


In [17]:
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import classification_report


df_development = pd.read_csv("../datasets/development.csv", sep=",")
df_evaluation = pd.read_csv("../datasets/evaluation.csv", sep=",")

df_development.fillna("", inplace=True)
df_evaluation.fillna("", inplace=True)

df_development["all_text"] =df_development["source"] + " " + df_development["title"] + " " + df_development["article"]
df_evaluation["all_text"] = df_evaluation["source"] + " " + df_evaluation["title"] + " " + df_evaluation["article"]

X_train_val = df_development["all_text"]
y_train_val = df_development["label"]


k = 5
skcv = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)

tfidf = TfidfVectorizer(
    max_features=120000, 
    max_df=0.7,
    min_df=3,
    stop_words="english",
    sublinear_tf=True,
    preprocessor=None,
    analyzer="word",
    ngram_range=(1,2)
    )

clf_lsvc = LinearSVC(
    max_iter=1000,
    C=0.2,
    random_state=42,
    class_weight="balanced"
)

lsvc_pipe = Pipeline([
    ("tfidf", tfidf),
    ("clf_lsvc", clf_lsvc)
])


y_pred_lsvc = cross_val_predict(
    estimator = lsvc_pipe,
    X = X_train_val,
    y = y_train_val,
    cv=skcv
)

labels = [0, 1, 2, 3, 4, 5, 6]
targets = {"label0", "label1", "label2","label3","label4","label5","label6"}

print(classification_report(y_train_val, y_pred_lsvc,labels=labels,target_names=targets))

              precision    recall  f1-score   support

      label3       0.76      0.75      0.76     23542
      label1       0.74      0.83      0.78     10588
      label5       0.80      0.85      0.83     11161
      label4       0.64      0.53      0.58      9977
      label0       0.79      0.96      0.86      8574
      label6       0.63      0.46      0.53     13053
      label2       0.57      0.84      0.68      3102

    accuracy                           0.73     79997
   macro avg       0.70      0.75      0.72     79997
weighted avg       0.72      0.73      0.72     79997



In [18]:
import joblib

lsvc_pipe.fit(X_train_val, y_train_val)
joblib.dump(lsvc_pipe, "final_model.joblib")

# Save the final model

['final_model.joblib']

In [19]:
import pandas as pd

model = joblib.load("final_model.joblib")
# Upload the saved model

X_evaluation = df_evaluation["all_text"]
# get the necessary features from evaluation dataframe

pred = model.predict(X_evaluation)
# prediction the using saved model

submission = pd.DataFrame({
    "Id": df_evaluation["Id"],
    "Predicted": pred
})
# creation of the dataframe in desired format

submission.to_csv("submission.csv", index=False)
# save the prediction into csv file